## Importing libs

In [3]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import yfinance as yf
from scipy import stats
from statsmodels.graphics.tsaplots import plot_acf
from statsmodels.stats.diagnostic import acorr_ljungbox, het_arch
from arch import arch_model
from datetime import datetime, timezone
from zoneinfo import ZoneInfo
from IPython.display import display

plt.rcParams['figure.figsize'] = (10, 4)
pd.set_option('display.max_columns', 12)

## 2 — Traduzir a proposta para Python
Um **dicionário** associa uma chave a um valor:  cada setor a uma lista de tickers. 
Um **ticker** identifica o título . 


In [13]:
setores = {
    'Information Technology': ['NVDA', 'AAPL', 'MSFT', 'AVGO', 'AMD', 'MU'],
    'Financials': ['BRK-B', 'JPM', 'V'],
    'Health Care': ['LLY', 'JNJ'],
    'Consumer Discretionary': ['AMZN', 'TSLA'],
    'Communication Services': ['T', 'GOOGL'],
    'Industrials': ['GE', 'CAT'],
    'Consumer Staples': ['WMT', 'COST'],
    'Energy': ['XOM'],
    'Utilities': ['NEE'],
    'Real Estate': ['WELL'],
    'Materials': ['LIN']
}

#mete as açoes todas numa lista extend pq com apend mete lista dentro de lista
acoes = []
for lista in setores.values():
    acoes.extend(lista)

print('Número de ações:', len(acoes))
print('Peso por ação:', f'{1 / len(acoes):.2%}')


# tabela  "peso_setorial" cad linha n_açoes por setor .Cada linha representa um setor.
peso_setorial = pd.DataFrame(

    {'n_acoes': [len(lista) for lista in setores.values()]
    },
    index=setores.keys() 
)
#print(peso_setorial)

# nova coluna com o peso de cada setor .
# Exemplo:
# se Tecnologias tem 6 ações e existem 23 ações no total:
# 100 * 6 / 23 = 26,09%

peso_setorial['peso_carteira_pct'] = (
    100 * peso_setorial['n_acoes'] / len(acoes)
)
#duas casas decimais.
peso_setorial.round(2)
print(peso_setorial)
inicio = '2010-01-01'  # Mais de 15 anos de histórico
fim = pd.Timestamp('2026-09-21')  # Inclui até 20/09/2026



referencias = ['SPY', '^GSPC']

tickers = acoes + referencias

print('Início:', inicio)
print('Fim:', fim.date())

Número de ações: 23
Peso por ação: 4.35%
                        n_acoes  peso_carteira_pct
Information Technology        6          26.086957
Financials                    3          13.043478
Health Care                   2           8.695652
Consumer Discretionary        2           8.695652
Communication Services        2           8.695652
Industrials                   2           8.695652
Consumer Staples              2           8.695652
Energy                        1           4.347826
Utilities                     1           4.347826
Real Estate                   1           4.347826
Materials                     1           4.347826
Início: 2010-01-01
Fim: 2026-09-21


## Descarregar primeiro uma  


In [14]:
exemplo = yf.Ticker('MSFT').history(
    start=inicio,
    end=fim.strftime('%Y-%m-%d'),
    auto_adjust=False,
    actions=True,
    repair=False
)



display(exemplo.head())
display(exemplo[['Close', 'Adj Close']].tail())

,Open,High,Low,Close,Adj Close,Volume,Dividends,Stock Splits
Date,,,,,,,,
2010-01-04 00:00:00-05:00,30.620001,31.100000,30.590000,30.950001,22.984236,38409100,0.0,0.0
2010-01-05 00:00:00-05:00,30.850000,31.100000,30.639999,30.959999,22.991663,49749600,0.0,0.0
2010-01-06 00:00:00-05:00,30.879999,31.080000,30.520000,30.770000,22.850563,58182400,0.0,0.0
2010-01-07 00:00:00-05:00,30.629999,30.700001,30.190001,30.450001,22.612928,50559700,0.0,0.0
2010-01-08 00:00:00-05:00,30.280001,30.879999,30.240000,30.660000,22.768873,51197400,0.0,0.0


,Close,Adj Close
Date,,
2026-09-14 00:00:00-04:00,505.410004,505.410004
2026-09-15 00:00:00-04:00,497.119995,497.119995
2026-09-16 00:00:00-04:00,490.299988,490.299988
2026-09-17 00:00:00-04:00,497.750000,497.750000
2026-09-18 00:00:00-04:00,493.779999,493.779999


## Repetir pra todas


In [ ]:
historicos = {}      # Tabelas completas: preços, volume, dividendos, etc.
series_precos = {}   # Apenas os preços escolhidos para cada ativo

# Percorrer cada ticker da nossa lista
for ticker in tickers:
    
    dados_ativo = yf.Ticker(ticker).history(
            start=inicio,
            end=fim.strftime('%Y-%m-%d'),     
            auto_adjust=False,                          # Manter Close e Adj Close separados
            actions=True,                               # Incluir dividendos e splits 
            repair=False                                # Não aplicar reparações automáticas do yfinance
        )


    # Retirar a informaçãoda data qe nao interessa       fuso horario
    dados_ativo.index = dados_ativo.index.tz_localize(None)

    # Colocar a hora em 00:00:00 para comparar os dados apenas por dia
    dados_ativo.index = dados_ativo.index.normalize()

    
    coluna_preco = 'Adj Close'  # Preço ajustado das ações e do SPY

    # Guardar a tabela completa 
    historicos[ticker] = dados_ativo

    # Guardar apenas preços 
    series_precos[ticker] = dados_ativo[coluna_preco]



# alinha os preços pelas datas.
precos = pd.DataFrame(series_precos)

# Ordenar 
precos = precos.sort_index()

precos.tail()

,NVDA,AAPL,MSFT,AVGO,AMD,MU,...,XOM,NEE,WELL,LIN,SPY,^GSPC
Date,,,,,,,,,,,,,
2026-09-14,210.960007,333.079987,505.410004,344.093414,493.410004,924.030029,...,165.080002,81.629997,234.210007,464.410004,758.995239,7619.979980
2026-09-15,212.169998,331.339996,497.119995,338.653320,504.200012,927.599976,...,169.320007,81.070000,232.759995,462.959991,755.513916,7585.729980
2026-09-16,213.899994,332.410004,490.299988,338.892914,512.500000,926.549988,...,163.320007,80.370003,234.500000,462.089996,752.182190,7551.810059
2026-09-17,219.339996,337.000000,497.750000,346.668732,545.090027,977.500000,...,163.270004,81.279999,232.259995,458.480011,760.710999,7637.759766
2026-09-18,222.270004,336.130005,493.779999,356.959991,559.820007,1015.799988,...,163.539993,80.470001,228.869995,460.399994,761.690002,7650.500000


## Verificar  calendário e observações em falta


In [17]:
# Usar como calendário  SPY
# remove valores em falta index obtém as respetivas datas.
calendario = series_precos['SPY'].dropna().index

precos = precos.reindex(calendario)

availability_rows = []

for ticker in tickers:

    # Selecionar os preços 
    observados = precos[ticker].dropna()

    # Encontrar a primeira e a última data com preço
    primeiro = observados.index.min()
    ultimo = observados.index.max()

    # Selecionar o intervalo entre essas duas datas na tabela original.
    # Aqui mantemos os NaN para detetar 
    intervalo = precos.loc[primeiro:ultimo, ticker]

    
    availability_rows.append({

        'ticker': ticker,

        # Primeira e última DATA  (SAO DATAS NAO PREÇOS)
        'primeiro_preco': primeiro,
        'ultimo_preco': ultimo,

        # Número de preços disponíveis
        'n_precos': len(observados),

        # Duração aproximada 
        'anos': (ultimo - primeiro).days / 365,

        # Número de preços em falta 
        'falta': intervalo.isna().sum(),

        # Número de sessões do calendário anteriores ao primeiro preço.
        'sessoes_antes_inicio': (calendario < primeiro).sum(),

        
    })

# Transformar a lista de dicionários em DATAFRAME
availability = pd.DataFrame(availability_rows).set_index('ticker')
    

availability.round(2)

,primeiro_preco,ultimo_preco,n_precos,anos,falta,sessoes_antes_inicio
ticker,,,,,,
NVDA,2010-01-04,2026-09-18,4203,16.72,0,0
AAPL,2010-01-04,2026-09-18,4203,16.72,0,0
MSFT,2010-01-04,2026-09-18,4203,16.72,0,0
AVGO,2010-01-04,2026-09-18,4203,16.72,0,0
AMD,2010-01-04,2026-09-18,4203,16.72,0,0
MU,2010-01-04,2026-09-18,4203,16.72,0,0
BRK-B,2010-01-04,2026-09-18,4203,16.72,0,0
JPM,2010-01-04,2026-09-18,4203,16.72,0,0
V,2010-01-04,2026-09-18,4203,16.72,0,0


## log retu8rns


In [ ]:
# Calcula os retornos logarítmicos diários aplicando o log aos preços 
# e subtraindo o valor do dia anterior (.diff())
retornos_diarios = np.log(precos).diff()


# exemplo 
retornos_diarios[['MSFT', 'T', 'SPY']].head()

,MSFT,T,SPY
Date,,,
2010-01-04,NaN,NaN,NaN
2010-01-05,0.000323,-0.004910,0.002644
2010-01-06,-0.006156,-0.009872,0.000704
2010-01-07,-0.010454,-0.011292,0.004212
2010-01-08,0.006873,-0.007353,0.003322


## Passar a semanas e meses


In [18]:
# Agrupa os preços por semana (com fim à sexta-feira)
# e seleciona a última linha disponível em cada semana.
precos_semanais = precos.resample('W-FRI').agg(lambda grupo: grupo.iloc[-1])

#igual mas com mes
precos_mensais = precos.resample('ME').agg(lambda grupo: grupo.iloc[-1])

# Define a data-limite para considerar apenas períodos já terminados.
limite_completo = pd.Timestamp('2026-09-20')

# Mantém apenas as semanas cuja data de fim é anterior ou igual ao limite.
precos_semanais = precos_semanais.loc[precos_semanais.index <= limite_completo]

# Assim, setembro de 2026 fica excluído, porque ainda não terminou.
precos_mensais = precos_mensais.loc[precos_mensais.index <= limite_completo]

# Calcula o retorno log

# shift(1) coloca, em cada linha, o preço anterior.
retornos_semanais = np.log(precos_semanais / precos_semanais.shift(1))

# Calcula o retorno logarítmico mensal usando o preço do mês anterior.
retornos_mensais = np.log(precos_mensais / precos_mensais.shift(1))

retornos_semanais.head()
retornos_mensais.tail()



,NVDA,AAPL,MSFT,AVGO,AMD,MU,...,XOM,NEE,WELL,LIN,SPY,^GSPC
Date,,,,,,,,,,,,,
2026-04-30,0.134814,0.066902,0.096765,0.299126,0.555358,0.425780,...,-0.094703,0.052434,0.094662,0.010794,0.099893,0.099151
2026-05-31,0.056356,0.140707,0.101216,0.067927,0.375620,0.629974,...,-0.053803,-0.117719,-0.053436,-0.006908,0.051288,0.050190
2026-06-30,-0.052589,-0.075524,-0.188149,-0.166230,0.118295,0.172914,...,-0.060590,0.015994,0.100199,0.044968,-0.010346,-0.010703
2026-07-31,0.003293,0.065378,0.219803,0.030066,-0.198863,-0.338094,...,0.128325,-0.009732,0.032382,-0.081383,0.000348,-0.001286
2026-08-31,0.095106,0.026241,0.089529,-0.049877,-0.011469,0.152617,...,0.041288,-0.046639,0.011417,0.023142,0.026447,0.025887


## Guardar na pasta data

In [ ]:
'''
import os

# Cria a pasta 
os.makedirs('data', exist_ok=True)


precos.to_csv('data/precos_limpos.csv')
retornos_diarios.to_csv('data/retornos_diarios.csv')
retornos_semanais.to_csv('data/retornos_semanais.csv')
retornos_mensais.to_csv('data/retornos_mensais.csv') 
'''